In [2]:
import fitz  # PyMuPDF is imported as fitz
import sys
import os


sys.path.append("../src") # Because by default Python only searches inside the current folder now it will also search in the src folder

In [3]:
pdf_path = "../src/data/Attention_is_all_you_need.pdf"

doc = fitz.open(pdf_path)

print("Number of pages", len(doc))

Number of pages 15


In [4]:
page = doc[0]
text = page.get_text()

text[:10000] # The beginning of the document

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence

In [5]:
from ingestion.loader import load_documents

docs = load_documents("../src/data") # Go to this folder and load all the documents inside it.

print(len(docs))

print(docs[0]["filename"])

print(docs[0]["text"][:1000])


4
Attention_is_all_you_need.pdf
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence 

In [6]:
from ingestion.chunker import chunk_documents

chunks = chunk_documents(docs)

print("Number of chunks: ", len(chunks))
print(chunks[0]["filename"])
print(chunks[0]["text"])

# Later each chunk will become a vector 

Number of chunks:  341
Attention_is_all_you_need.pdf
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [7]:
from ingestion.embeddings import create_embeddings

embeddings = create_embeddings(chunks)

print(embeddings.shape) 

# The model will convert each chunk into vector
# output --> (number of chunks, size / number of each vector)

c:\Users\yazan\anaconda3\envs\ml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6463.49it/s]


(341, 384)


For example, imagine you have:
```
texts = [
    "Traffic cameras improve road safety",
    "AI helps detect vehicles",
    "Road sensors reduce accidents"
]
```
- We have 3 chunks.


- The model converts each one:
```
Chunk 1
"Traffic cameras improve road safety"
        ↓
[0.21, 0.54, -0.32, ...]


Chunk 2
"AI helps detect vehicles"
        ↓
[0.15, 0.67, -0.11, ...]


Chunk 3
"Road sensors reduce accidents"
        ↓
[0.44, 0.23, 0.91, ...]
```

- The model ```all-MiniLM-L6-v2``` creates the vector with 384 numbers.

- So each chunk becomes:
```
Chunk 1 → [384 numbers]
Chunk 2 → [384 numbers]
Chunk 3 → [384 numbers]
```


- Your embeddings look like a matrix:


```
[
 [0.21, 0.54, -0.32, ..., 0.11],   ← Chunk 1 vector
 [0.15, 0.67, -0.11, ..., 0.55],   ← Chunk 2 vector
 [0.44, 0.23, 0.91, ..., 0.88]    ← Chunk 3 vector
]
```

The shape is:
```
(3, 384) 
```

Meaning:
```
3    → number of chunks
384  → numbers inside each vector
```

---




See [Checkpoint 1](../notes/Checkpoint-1.md).

In [ ]:
from retrieval.vector_store import store_embeddings, search_documents
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")


collection = store_embeddings(chunks, embeddings)
print("Stored successfully")



question = "YOLO version 8 VS YOLO version 11"

# First convert to vector
question_embedding = model.encode(question)

# Second search in the chromaDB
results = search_documents(question_embedding)
print(results["documents"])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3660.45it/s]


Stored successfully
[['ce Benchmarks: Comparative analyses reveal YOLO11’s superior performance, particularly in its\nsmaller variants. The nano model, despite a slight increase in parameters, demonstrates enhanced inference\nspeed and frames per second (FPS) compared to its predecessor. This improvement suggests that YOLO11\nachieves a favorable balance between computational efficiency and detection accuracy.\n6. Implications for Real-World Applications: The advancements in YOLO11 have significant implications\nfor variou', 'ing upon its predecessors while\nintroducing innovative enhancements. This latest iteration demonstrates remarkable versatility and efficiency across\nvarious CV tasks.\n1. Efficiency and Scalability: YOLO11 introduces a range of model sizes, from nano to extra-large, catering\nto diverse application needs. This scalability allows for deployment in scenarios ranging from resource-\nconstrained edge devices to high-performance computing environments. The nano varia

Step 1
results["documents"]

returns:

[
    [
*        "Traffic cameras improve road safety.",
*        "AI can detect speeding vehicles.",
*        "Camera placement affects accuracy."
    ]
]

In [13]:
from generation.llm import generate_answer

# See above example
# the restuls will be a list of lists, we need to get the first list which contains the text
# Then "\n" --> joins the list into one string.
context = "\n".join(results["documents"][0])

# Call the LLM
answer = generate_answer(question, context)
print(answer)


# Show the metadatas

print("\nSources:")

sources = set() # To remove duplicte sources

# Loop over each metadata in the metatdatas array, which comes from the results of searching the documents in database
for metadata in results["metadatas"][0]:
    if metadata: # If it exists
        sources.add(metadata["filename"]) # Add the file name to the set

for source in sources:
    print("-", source)


 The provided context does not directly compare YOLO version 8 and YOLO version 11. However, it does mention that YOLO11 demonstrates superior performance compared to its predecessor (implied to be YOLOv7 or earlier versions). It also mentions the introduction of a range of model sizes in YOLO11, including a nano variant that shows impressive results on various benchmarks. As for developer-centric improvements, it suggests that YOLOv8 offers compatibility with Darknet and PyTorch frameworks, and an enhanced user experience through its Python API and command-line interface. Without more specific information, I cannot provide a detailed comparison between the two versions.

Sources:
- YOLOv11.pdf
- YOLOv8.pdf


- See [Note](../src/notes/chromadb-rag-query-results.md) to understnad the restuls structure.